# pyblt — viewing rblt bio-logger files in Python (VS Code)

Opens the `.h5` files produced by the R package [rblt](https://github.com/sg4r/rblt) (Axytrek, CATS, LUL, WACU, LDATA) as synchronised, zoomable plots, with BORIS behaviour labels shaded on top.

**Setup:** keep `pyblt.py` in the same folder as this notebook. In VS Code, install the *Python* and *Jupyter* extensions, then pick your Python environment with **Select Kernel** (top right).

*Based on rblt by Sébastien Geiger (CNRS IPHC, GPL ≥ 3).*

## 1. Install packages
Installs into the kernel you selected. Run once per environment, then restart the kernel (⟳ Restart in the notebook toolbar).

In [1]:
%pip install -q numpy pandas h5py plotly plotly-resampler ipywidgets anywidget

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


## 2. Import pyblt

In [2]:
%load_ext autoreload
%autoreload 2
# autoreload: edits you make to pyblt.py are picked up without restarting the kernel
import pyblt

## 3. Point to your data
Paths can be absolute or relative to this notebook's folder. The behaviour file is optional: a BORIS *aggregated events* CSV (`Behavior`, `Start (s)`, `Stop (s)`, in seconds from the logger start). Set `BORIS_FILE = None` if you have none.

In [3]:
from pathlib import Path

H5_FILE    = Path(r"C:\Users\loren\Documents\Tortues\Martinique_Verte\fichiers_h5\Labelled_dataset\CC-07-47_14-02-2018.h5")       # <- edit
BORIS_FILE = Path(r"C:\Users\loren\Documents\Tortues\Martinique_Verte\Behavior\Behaviors_CC-07-47_14-02-2018.csv")  # <- edit, or None
SEP        = ";"                             # separator of the BORIS file

lg = pyblt.load(str(H5_FILE), str(BORIS_FILE) if BORIS_FILE else None, sep=SEP)
lg.summary()

C:\Users\loren\Documents\Tortues\Martinique_Verte\fichiers_h5\Labelled_dataset\CC-07-47_14-02-2018.h5
  logger: LDATA (v0.1.0), source: CC-07-47_14-02-2018_cut_pression.h5
  start: 2018-02-14 14:05:31+00:00 UTC, rate: 20 Hz, rows: 126,000, duration: 0 days 01:45:00
  metrics: Accelerometer, Gyroscope, Pression
  behaviours: Swimming horizontally (45), Swimming ascent (38), Swimming descent (34), Breathing (14), Left U-turn (5), Right U-turn (2), Swimming on the bottom (2), Time mark (1)


## 4. Interactive viewer
Choose metrics and behaviours (Ctrl/Cmd-click for several), set a time window, press **Plot**. Drag on a panel to zoom; all panels follow and the data re-samples at full resolution. Double-click to zoom out.

In [4]:
pyblt.viewer(lg)

# If no plot appears below the controls, use the static version
# (always displays, but does not re-sample when you zoom):
# pyblt.viewer(lg, widget=False)

## 5. Plot directly from code

In [ ]:
fig = pyblt.plot(lg,
                 metrics=["Accelerometer"],   # names shown by lg.summary()
                 behaviors=None,              # None = all behaviours
                 start=None, end=None)        # e.g. "2018-02-14 14:30"
fig

In [ ]:
# Static fallback, or to save a figure as an interactive HTML file
static = pyblt.plot(lg, widget=False)
static.write_html("pyblt_plot.html")
static.show()

## 6. Work with the data
`lg.data` is a pandas DataFrame indexed by UTC time; `lg.behaviors` holds the behaviour bouts.

In [ ]:
lg.data.head()

In [ ]:
per_second = lg.data.resample("1s").mean()

if lg.behaviors is not None:
    first = lg.behaviors.iloc[0]
    bout = lg.data.loc[first["start"]:first["stop"]]
    print(first["Behavior"], bout.shape)